# Day 2 Project — Engineering Knowledge Assistant

The complete system combines local document processing and embeddings with hosted generation:

```text
Documents → chunks → local embeddings → Chroma retrieval
→ evidence context → OpenRouter answer → citations/abstention
```

Retrieval and answer behaviour are evaluated separately.

## Before you begin

### Learning outcomes

Integrate ingestion, retrieval, citations, abstention, state, and separate evaluation.

Architecture reference: [D06–D07](../../diagrams/source/day_02.md).

### Expected observation

The ten-case report exposes retrieval and answer outcomes rather than one vague score.


## Concept briefing

## What to carry into Day 3

Knowledge usually comes from an external corpus. Memory usually records selected
information from interactions. Neither should be confused with active context. Day 3
shows how history grows, why summaries lose information, and how persistent memory and
execution policy require explicit lifecycle controls.


In [ ]:
import os,sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
here=Path.cwd().resolve(); candidates=[here,here/"day_02_knowledge_and_state",here.parent]
project_root=next(p for p in candidates if (p/"src"/"knowledge_agent").exists())
sys.path.insert(0,str(project_root/"src")); sys.path.insert(0,str(project_root))
from knowledge_agent.evaluation import evaluate_answers,evaluate_retrieval,load_golden_set,summarize
from run_project import build_assistant

## Build the classroom assistant

Classroom mode uses Sentence Transformers, Chroma, and OpenRouter. Use mock mode only to debug the surrounding pipeline without network/model downloads.

In [ ]:
USE_MOCK=False
assistant=build_assistant("mock" if USE_MOCK else "classroom")
state=assistant.answer("Does requesting island mode immediately open the grid breaker?")
print(state.answer.model_dump_json(indent=2) if state.answer else state.error)

## Inspect execution state

In [ ]:
print("status:",state.status)
for item in state.retrieved:
    print(item.rank,round(item.score,3),item.chunk.source,item.chunk.section)
print("citations:",[c.model_dump() for c in state.answer.citations])

## Run the 10-case evaluation

This makes real API calls in classroom mode. Keep the set small and do not rerun it unnecessarily.

In [ ]:
cases=load_golden_set(project_root/"data"/"golden_set.json")
retrieval=evaluate_retrieval(assistant.index,cases,top_k=3)
answers=evaluate_answers(assistant,cases)
print("retrieval:",summarize(retrieval,["source_hit","section_hit"]))
print("answers:",summarize(answers,["completed","abstention_correct","citation_correct"]))

## Diagnose, do not guess

For each failed case decide: ingestion failure, retrieval failure, insufficient evidence, generation failure, citation failure, or evaluation-definition problem. Change one layer and rerun the same cases.

## Final reflection

Explain: why RAG is not training; why top-k is a trade-off; why citations need validation; how state differs from context and memory; and when retrieval should be deterministic versus an agent tool.

Day 2 gave the agent **knowledge**. Day 3 handles growing context, persistent memory, planning, permissions, approval, and observability.

## Your turn

Diagnose one missed case using query, chunks, expected source, and proposed change.

## Recap

A knowledge agent is only as reliable as its retrieval evidence and evaluation.
